# Figuring out BCPNN

In [ ]:
from vigipy import *
import pandas as pd

import duckdb
import sys
sys.path.insert(0, "..")  # per trovare il modulo vigipy locale

Overall structure of the data:

In [ ]:
from pyarrow.parquet import ParquetFile
import pyarrow as pa 

n=100000

pf = ParquetFile(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat_deduped.parquet") 
first_n_rows = next(pf.iter_batches(batch_size = n)) 
ae_df = pa.Table.from_batches([first_n_rows]).to_pandas() 

In [ ]:
[name for name in ae_df.columns]

import sys
sys.path.insert(0,'..')

In [ ]:
from vigipy import GPS
from vigipy.utils import Container
from src.contingency_table import build_contingency_table, qc_contingency_table

ct = pd.read_parquet("data/contingency_table.parquet")

def contingency_to_vigipy(ct: pd.DataFrame) -> tuple[Container, int]:
    """
    Converte la contingency table 2x2 nel formato atteso da vigipy GPS.
    """
    df = ct.copy()
    df = df.rename(columns={
        "drug": "product_name",
        "pt":   "ae_name",
        "a":    "events",
    })
    df["product_aes"]          = df["events"] + df["b"]   # a + b
    df["count_across_brands"]  = df["events"] + df["c"]   # a + c

    N = int(df["n"].iloc[0])  # totale unico per tutto il sottoinsieme

    container = Container(params=False)
    container.data = df[["product_name", "ae_name", "events",
                          "product_aes", "count_across_brands"]]
    container.N    = N

    # Matrice di contingenza pivot (drug x PT) — serve per stimare i prior
    container.contingency = df.pivot_table(
        index="product_name",
        columns="ae_name",
        values="events",
        fill_value=0
    )

    return container, N

container, N = contingency_to_vigipy(ct)
print(f"N totale report: {N}")
print(f"Coppie drug-PT : {len(container.data)}")
container.data.head()


Funziona! caccio dentro bcpnn, the article on Iapatinib sets IC>0, lower limit of CI>0, N>=3

In [ ]:
res=bcpnn(container=container, min_events=3, decision_metric="rank", ranking_statistic="quantile")
# figure out cosa devo fare per fare quello che hanno fatto per iapatinib

In [ ]:
res.all_signals

# bisogna trovare i segnali rilevanti quali sono 
